In [1]:
# Python 3.13.1

In [13]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import numpy as np

# 경로 설정
TRAIN_DIR = r"/Users/ihaeni/dev/ai/computervision/data/train"
TEST_DIR = r"/Users/ihaeni/dev/ai/computervision/data/test"
BATCH_SIZE = 8 

# Train용 전처리 (Augmentation 포함)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Validation/Test용 전처리 (단순 Resize)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 전체 train dataset 불러오기
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)

# train:val = 8:2 split
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# test dataset
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=test_transform)

# 클래스 별 갯수 확인
print(Counter(full_train_dataset.targets))

# Sampler 설정
train_indices = train_dataset.indices
train_targets = [full_train_dataset.targets[i] for i in train_indices]

from collections import Counter
class_counts = Counter(train_targets)
class_sample_count = np.array([class_counts[i] for i in range(len(class_counts))])
weights = 1. / class_sample_count
samples_weight = [weights[label] for label in train_targets]
sampler = WeightedRandomSampler(samples_weight, num_samples=len(samples_weight), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 클래스 이름 확인
print("Classes:", full_train_dataset.classes)


Counter({1: 137, 3: 76, 5: 74, 0: 70, 2: 70, 4: 70, 6: 70, 7: 70, 8: 70, 9: 70})
Classes: ['1', '10', '2', '3', '4', '5', '6', '7', '8', '9']


In [14]:
# 배치 하나 꺼내기
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

# 예측하기
with torch.no_grad():
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)

# 결과 출력
print("예측 인덱스:", preds)
print("정답 인덱스:", labels)


예측 인덱스: tensor([2, 1, 9, 5, 0, 2, 4, 1])
정답 인덱스: tensor([2, 1, 9, 5, 0, 2, 4, 1])


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# 클래스 수 확인
num_classes = len(full_train_dataset.classes)

# ResNet18 불러오기 (pretrained)
from torchvision import models
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 마지막 fc 교체
model.fc = nn.Linear(model.fc.in_features, num_classes)

# 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 학습 기록용 리스트
train_accs, val_accs = [], []
train_losses, val_losses = [], []
best_val_acc = 0.0
best_train_acc = 0.0
best_model_state = None

# 학습 루프
for epoch in range(10):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        loss_sum += loss.item()

    scheduler.step()
    train_acc = 100 * correct / total
    train_accs.append(train_acc)
    train_losses.append(loss_sum / len(train_loader))

    # 검증
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    val_acc = 100 * correct / total
    val_accs.append(val_acc)
    val_losses.append(val_loss_sum / len(val_loader))

    print(f"[Epoch {epoch+1}] Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

    # Best 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_train_acc = train_acc 
        best_model_state = model.state_dict()
        print("✅ Best model updated!")

# 학습 완료 후 Best 모델 로드
model.load_state_dict(best_model_state)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

[Epoch 1] Train Acc: 32.69%, Val Acc: 35.26%
✅ Best model updated!
[Epoch 2] Train Acc: 48.47%, Val Acc: 33.97%
[Epoch 3] Train Acc: 54.59%, Val Acc: 42.31%
✅ Best model updated!
[Epoch 4] Train Acc: 60.87%, Val Acc: 62.18%
✅ Best model updated!
[Epoch 5] Train Acc: 63.12%, Val Acc: 67.31%
✅ Best model updated!
[Epoch 6] Train Acc: 77.62%, Val Acc: 85.90%
✅ Best model updated!
[Epoch 7] Train Acc: 84.54%, Val Acc: 85.26%
[Epoch 8] Train Acc: 86.31%, Val Acc: 89.74%
✅ Best model updated!
[Epoch 9] Train Acc: 92.11%, Val Acc: 88.46%
[Epoch 10] Train Acc: 93.40%, Val Acc: 89.74%
Best Validation Accuracy: 89.74%


In [7]:
# 배치 하나 꺼내기(학습 후)
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# 예측하기
with torch.no_grad():
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)

# 결과 출력
print("예측 인덱스:", preds)
print("정답 인덱스:", labels)

예측 인덱스: tensor([1, 0, 0, 0, 0, 0, 0, 0])
정답 인덱스: tensor([0, 0, 0, 0, 0, 0, 0, 0])


In [12]:
# Test 데이터셋으로 최종 성능 평가
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Train / Val / Test Accuracy를 단순 Print
test_acc = 100 * correct / total
print(f"Final Accuracy - Train: {best_train_acc:.2f}%, Val: {best_val_acc:.2f}%, Test: {test_acc:.2f}%")

Final Accuracy - Val: 89.74%, Test: 77.58%


In [ ]:
# 학습 과정 시각화
import matplotlib.pyplot as plt
epochs = range(1, len(train_accs) + 1)

plt.figure(figsize=(12, 6))

# Loss 그래프
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()